### Analysis of the GitHub Actions Workflow Run Data Extraction Software

This cell documents the operation of the `script.py` script, whose purpose is to automate the collection of information about GitHub Actions executions (*workflow runs*) from multiple public repositories.

#### General Software Purpose

This software allows querying the GitHub API to obtain detailed information about the execution of CI/CD workflows from a list of repositories. For each execution found, the following are downloaded and stored:

1. General execution metadata (`workflow_run.json`).
2. Details of the jobs that compose it (`jobs.json`).
3. The workflow YAML file that defines the execution.
4. Compressed logs (`logs.zip`).

All of this is saved in an organized manner in folders by repository and execution, allowing the creation of a dataset for subsequent analysis, comparisons between executions, failure studies, etc.

#### How does the software work?

1. **Repository reading:**
   - The script starts from a `repos.csv` file, where each row defines a repository in `owner,repo` format.
   - It iterates through this list to process the repositories one by one.

2. **GitHub API Connection:**
   - It uses the personal token `GITHUB_TOKEN` (obtained from `.env`) to authenticate requests.

3. **Workflow runs extraction:**
   - For each repository, up to `MAX_RUNS` recent executions are downloaded.
   - It queries `/actions/runs` and paginates if necessary.

4. **Processing per execution:**
   - For each execution, the software does the following:
     - It queries its details with `/runs/{run_id}`.
     - It retrieves the associated jobs with `/runs/{run_id}/jobs`.
     - It downloads the workflow YAML file if possible.
     - It downloads the compressed logs.
   - The files are saved under a folder named with a sanitized `run_id + workflow name` (the name must be sanitized to avoid execution errors caused by creating files with disallowed characters or lengths).

5. **Additional classification:**
   - If an execution ended in **failure**, it is also copied to the `failure_workflow_runs` folder.
   - If it is a **rerun** (i.e., `run_attempt > 1`), it is also copied to `retry_workflow_runs`.

6. **Controlled pause:**
   - The script includes a wait (`time.sleep(1)`) between executions to avoid hitting the GitHub API limit and prevent throttling errors.

#### Information Organization

The information is stored under a folder per repository (with the name format "owner_repo"), which in turn contains:

- `all_workflow_runs/`: All executions.
- `failure_workflow_runs/`: Only failed executions.
- `retry_workflow_runs/`: Only retries.

Each execution has its own subfolder with all the mentioned data.

#### Important Notes

- The script handles network errors and invalid API responses with informative messages and continues with the next repository or execution.
- Folder and file names are sanitized to avoid invalid characters.
- Functions are organized modularly to facilitate maintenance or reuse.

This cell marks the point prior to the execution of the software script. Next, we will execute `script.py` to begin data collection.

In [ ]:
!python script.py